In [2]:
import carla
import math
import random
import time
import numpy as np
import cv2

client = carla.Client('localhost', 2000)
client.reload_world()
world = client.get_world()

bp_lib = world.get_blueprint_library()
spawn_points = world.get_map().get_spawn_points()

In [22]:
vehicle_bp = bp_lib.find('vehicle.tesla.model3')
vehicle = world.try_spawn_actor(vehicle_bp, random.choice(spawn_points))
camera_bp = bp_lib.find('sensor.camera.rgb')
camera_bp.set_attribute('image_size_x', '1600')
camera_bp.set_attribute('image_size_y', '900')
camera_bp.set_attribute('fov', '70')

# 6 camera suite setup
spectator = world.get_spectator()

camera_f_trans = carla.Transform(carla.Location(z=1.5))                            # front middle
camera_fl_trans = carla.Transform(carla.Location(z=1.5), carla.Rotation(yaw=-55))  # front left
camera_fr_trans = carla.Transform(carla.Location(z=1.5), carla.Rotation(yaw=+55))  # front right
camera_b_trans = carla.Transform(carla.Location(z=1.5), carla.Rotation(yaw=+180))  # back middle
camera_bl_trans = carla.Transform(carla.Location(z=1.5), carla.Rotation(yaw=-120)) # back left
camera_br_trans = carla.Transform(carla.Location(z=1.5), carla.Rotation(yaw=+120)) # back right

In [25]:
# FRONT camera spawn
f_camera = world.spawn_actor(camera_bp, camera_f_trans, attach_to=vehicle)

time.sleep(2)
spectator.set_transform(f_camera.get_transform())
#f_camera.destroy()

In [ ]:
# FRONT LEFT camera spawn
fl_camera = world.spawn_actor(camera_bp, camera_fl_trans, attach_to=vehicle)

time.sleep(2)
spectator.set_transform(fl_camera.get_transform())
fl_camera.destroy()

In [ ]:
# FRONT RIGHT camera spawn
fr_camera = world.spawn_actor(camera_bp, camera_fr_trans, attach_to=vehicle)

time.sleep(2)
spectator.set_transform(fr_camera.get_transform())
fr_camera.destroy()

In [ ]:
# BACK MIDDLE camera spawn
b_camera = world.spawn_actor(camera_bp, camera_b_trans, attach_to=vehicle)

time.sleep(2)
spectator.set_transform(b_camera.get_transform())
b_camera.destroy()

In [ ]:
# BACK LEFT camera spawn
bl_camera = world.spawn_actor(camera_bp, camera_bl_trans, attach_to=vehicle)

time.sleep(2)
spectator.set_transform(bl_camera.get_transform())
bl_camera.destroy()

In [29]:
# BACK RIGHT camera spawn
# remember, this camera has 110 fov

br_camera_bp = bp_lib.find('sensor.camera.rgb')
br_camera_bp.set_attribute('image_size_x', '1600')
br_camera_bp.set_attribute('image_size_y', '900')
br_camera_bp.set_attribute('fov', '110')
br_camera = world.spawn_actor(br_camera_bp, camera_br_trans, attach_to=vehicle)

time.sleep(2)
spectator.set_transform(br_camera.get_transform())
#br_camera.destroy()

In [6]:
def camera_callback(image, data_dict):
    data_dict['image'] = np.reshape(np.copy(image.raw_data), (image.height, image.width, 4))

In [26]:
image_w = camera_bp.get_attribute('image_size_x').as_int()
image_h = camera_bp.get_attribute('image_size_y').as_int()

camera_data = {'image': np.zeros((image_w, image_h, 4))}

# need to set the camera to the correct one, in the future need to all 6 at the same time
# doing front middle for now
f_camera.listen(lambda image: camera_callback(image, camera_data))

In [26]:
# for testing back right camera fov

image_w = camera_bp.get_attribute('image_size_x').as_int()
image_h = camera_bp.get_attribute('image_size_y').as_int()

camera_data = {'image': np.zeros((image_w, image_h, 4))}

# need to set the camera to the correct one, in the future need to all 6 at the same time
# doing front middle for now
f_camera.listen(lambda image: camera_callback(image, camera_data))

In [27]:
vehicle.set_autopilot(True)

In [28]:
window_name = 'RGB Camera'
cv2.namedWindow(window_name, cv2.WINDOW_AUTOSIZE)
cv2.imshow(window_name, camera_data['image'])
cv2.waitKey(1)

while True:
    cv2.imshow(window_name, camera_data['image'])

    if cv2.waitKey(1) == ord('q'):
        break

cv2.destroyAllWindows()

In [20]:
print(camera_bp.get_attribute('fov'))

ActorAttribute(id=fov,type=float,value=90)
